<div align='center'>

# 🚀 ComfyUI Ultimate Colab

**The most powerful, production-quality Google Colab launcher for ComfyUI**

[![GitHub](https://img.shields.io/badge/GitHub-ComfyUI--Ultimate--Colab-blue?logo=github)](https://github.com/TURBO-gif/ComfyUI-Ultimate-Colab)
[![Python](https://img.shields.io/badge/python-3.12-blue.svg)](https://www.python.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Instructions**: Run each cell in order from top to bottom.  
> Use `Runtime → Run all` for a fully automated setup.

</div>


---
## 📂 Cell 1 — Mount Google Drive

Mounts your Google Drive so that models, outputs, and workflows persist across sessions.

In [ ]:
import sys
import os

# ── Clone the launcher if not already present ──────────────────────────────────
LAUNCHER_DIR = "/content/ComfyUI-Ultimate-Colab"
LAUNCHER_REPO = "https://github.com/TURBO-gif/ComfyUI-Ultimate-Colab.git"

if not os.path.exists(LAUNCHER_DIR):
    print("Cloning ComfyUI Ultimate Colab ...")
    os.system(f"git clone --depth=1 {LAUNCHER_REPO} {LAUNCHER_DIR}")
else:
    print(f"Launcher already present at {LAUNCHER_DIR}")

# Add launcher to path
if LAUNCHER_DIR not in sys.path:
    sys.path.insert(0, LAUNCHER_DIR)

# ── Bootstrap logging + config ─────────────────────────────────────────────────
from comfy_launcher.config import get_config
from comfy_launcher.logger import setup_logging
from comfy_launcher.paths import get_paths

cfg = get_config(f"{LAUNCHER_DIR}/config.json")
setup_logging(level=cfg.log_level)
paths = get_paths(cfg)

# ── Mount Google Drive ─────────────────────────────────────────────────────────
from comfy_launcher.drive import DriveManager

drive_mgr = DriveManager(cfg, paths)
mounted = drive_mgr.mount()

if mounted:
    print(f"\n✅ Google Drive mounted")
    print(f"   Drive root: {cfg.drive_root}")
else:
    print("ℹ️  Running without Google Drive (changes won't persist)")


---
## 📦 Cell 2 — Install Dependencies

Installs the `comfy_launcher` package and its dependencies.

In [ ]:
import subprocess, sys

print("Installing comfy_launcher dependencies ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{LAUNCHER_DIR}/requirements.txt"],
    check=True
)
print("✅ Dependencies installed")


---
## ⚙️ Cell 3 — Install or Update ComfyUI

Clones ComfyUI if not present, or pulls the latest updates.

In [ ]:
from comfy_launcher.installer import Installer

installer = Installer(cfg, paths)

if installer.is_installed:
    print(f"ComfyUI found at {paths.comfyui_dir} — updating ...")
    installer.update()
else:
    print(f"Installing ComfyUI to {paths.comfyui_dir} ...")
    installer.install()

version = installer.get_comfyui_version()
print(f"\n✅ ComfyUI ready  |  Version: {version}")


---
## 🧩 Cell 4 — Install Custom Nodes

Installs ComfyUI Manager, Impact Pack, Efficiency Nodes, and any other nodes configured in `config.json`.

In [ ]:
from comfy_launcher.node_manager import NodeManager

node_mgr = NodeManager(cfg, paths)

# Install all nodes defined in config.json
node_mgr.install_defaults()

print("\n📋 Installed custom nodes:")
node_mgr.print_inventory()


---
## 📋 Cell 5 — Install Node Requirements

Installs Python requirements for all custom nodes.

In [ ]:
from comfy_launcher.utils import pip_install_requirements

nodes_dir = paths.comfyui_custom_nodes_dir
if nodes_dir.exists():
    for node_dir in nodes_dir.iterdir():
        req_file = node_dir / "requirements.txt"
        if req_file.exists():
            print(f"Installing requirements for {node_dir.name} ...")
            pip_install_requirements(req_file)

print("\n✅ All node requirements installed")


---
## 🔗 Cell 6 — Link Models

Symlinks Google Drive model directories into ComfyUI so models load directly from Drive.

In [ ]:
if drive_mgr.is_mounted:
    drive_mgr.ensure_directories()
    drive_mgr.link_models()
    drive_mgr.link_outputs()
    drive_mgr.link_custom_nodes()
    print("✅ Google Drive directories linked to ComfyUI")
else:
    print("ℹ️  Drive not mounted — using local model directories")
    paths.ensure_comfyui_dirs()
    print(f"   Model dirs created at {paths.comfyui_models_dir}")


---
## ⬇️ Cell 6.5 — (Optional) Download Models

Edit the `MODELS_TO_DOWNLOAD` list below to download models before launching.

Supports HuggingFace, CivitAI, GitHub, and direct URLs.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Add URLs to download. Leave empty to skip.
# Format: {"url": "...", "type": "loras"} or just {"url": "..."} for auto-detect.
# ──────────────────────────────────────────────────────────────────────────────
MODELS_TO_DOWNLOAD = [
    # Example Flux model:
    # {"url": "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors"},
    
    # Example LoRA from CivitAI:
    # {"url": "https://civitai.com/models/133005"},
    
    # Example direct URL:
    # {"url": "https://example.com/mymodel.safetensors", "type": "checkpoints"},
]

if MODELS_TO_DOWNLOAD:
    from comfy_launcher.model_manager import ModelManager
    model_mgr = ModelManager(cfg, paths)
    downloaded = model_mgr.download_batch(MODELS_TO_DOWNLOAD)
    print(f"\n✅ Downloaded {len(downloaded)} model(s)")
    model_mgr.print_inventory()
else:
    print("ℹ️  No models configured for download — add URLs to MODELS_TO_DOWNLOAD above")


---
## 📊 Cell 7 — Dashboard

Shows GPU, RAM, disk, ComfyUI version, installed models and nodes.

In [ ]:
from comfy_launcher.dashboard import Dashboard
from comfy_launcher.model_manager import ModelManager
from comfy_launcher.node_manager import NodeManager

model_mgr = ModelManager(cfg, paths)
node_mgr = NodeManager(cfg, paths)

models = model_mgr.scan_disk()
nodes = node_mgr.scan_disk()

dash = Dashboard(cfg, paths)
dash.render_once(
    comfyui_version=installer.get_comfyui_version(),
    model_count=len(models),
    node_count=len(nodes),
)


---
## 🌐 Cell 8 — Start Cloudflare Tunnel

Starts a Cloudflare Tunnel to make ComfyUI accessible from anywhere.

Change `TUNNEL_PROVIDER` to `"pinggy"` or `"localtunnel"` if desired.

In [ ]:
from comfy_launcher.tunnel import TunnelManager

# ── Configuration ──────────────────────────────────────────────────────────────
TUNNEL_PROVIDER = "cloudflare"  # Options: cloudflare | pinggy | localtunnel
# ──────────────────────────────────────────────────────────────────────────────

tunnel_mgr = TunnelManager(cfg, paths)
tunnel_url = tunnel_mgr.start(provider=TUNNEL_PROVIDER)

if tunnel_url:
    print(f"\n🌐 Tunnel active: {tunnel_url}")
    print(f"   Open this URL in your browser to access ComfyUI")
else:
    print("⚠️  Could not start tunnel — ComfyUI will be local-only")
    tunnel_url = None


---
## 🚀 Cell 9 — Launch ComfyUI

Starts the ComfyUI server. Access it via the tunnel URL above.

In [ ]:
from comfy_launcher.launcher import Launcher
from comfy_launcher.dashboard import Dashboard
from rich.console import Console

console = Console()

launcher = Launcher(cfg, paths)
proc = launcher.start(background=True)

if proc:
    console.print("\n[bold green]✅ ComfyUI is starting ...[/]")
    
    # Wait for ComfyUI to be ready
    ready = launcher.wait(timeout=120)
    
    if ready:
        console.print(f"[bold green]🎉 ComfyUI is ready![/]")
        if tunnel_url:
            console.print(f"\n[bold cyan]🌐 Open ComfyUI at:[/] [link={tunnel_url}]{tunnel_url}[/link]\n")
        else:
            console.print(f"[bold cyan]🌐 Local access:[/] http://localhost:{cfg.comfyui_port}\n")
        
        # Show final dashboard
        dash = Dashboard(cfg, paths)
        dash.render_once(
            tunnel_url=tunnel_url,
            comfyui_version=installer.get_comfyui_version(),
            model_count=len(models),
            node_count=len(nodes),
        )
    else:
        console.print("[bold yellow]⚠️  ComfyUI may still be starting — check the logs[/]")
else:
    console.print("[bold red]❌ Failed to start ComfyUI — run Cell 3 first[/]")


---
## 🛠️ Utilities

Run these cells individually as needed.

In [ ]:
# ── List installed models ──────────────────────────────────────────────────────
from comfy_launcher.model_manager import ModelManager
ModelManager(cfg, paths).print_inventory()


In [ ]:
# ── List installed nodes ───────────────────────────────────────────────────────
from comfy_launcher.node_manager import NodeManager
NodeManager(cfg, paths).print_inventory()


In [ ]:
# ── Create a backup ────────────────────────────────────────────────────────────
from comfy_launcher.backup import BackupManager
backup_mgr = BackupManager(cfg, paths)
manifest = backup_mgr.create(description="Manual backup")
print(f"\n✅ Backup created: {manifest.id} ({manifest.size_human})")
backup_mgr.print_list()


In [ ]:
# ── Update ComfyUI + nodes ─────────────────────────────────────────────────────
from comfy_launcher.updater import Updater
updater = Updater(cfg, paths)
results = updater.update_all()
print(f"\nComfyUI updated: {results['comfyui_updated']}")
updated_nodes = [k for k, v in results['nodes'].items() if v]
print(f"Updated nodes: {updated_nodes or 'none'}")


In [ ]:
# ── Restart tunnel ─────────────────────────────────────────────────────────────
new_url = tunnel_mgr.restart()
tunnel_url = new_url
if new_url:
    print(f"\n🌐 New tunnel URL: {new_url}")
